In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp


import os
import sys
import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd
import gc

import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *
from analysis_village.cc1pi.var_configs import *



from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks import CutMasks
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

In [ ]:
from cols_to_keep import *
keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
bnb_path = "/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p_pruned_for_data_mc_comp.df"
mc_bnb_df = load_df(bnb_path, keys2load, 10, filter_df = False, reprocess_df = False, reprocess_truth = False)
mc_bnb_evt_df = mc_bnb_df['cc1pi']


#mc_bnb_nu_df = mc_bnb_df['nudf']
mc_bnb_hdr_df = mc_bnb_df['hdr']
cols_to_keep = truth_cols_to_keep_slim
#mc_bnb_nu_df = mc_bnb_nu_df[cols_to_keep]

del mc_bnb_df
gc.collect()

In [ ]:
print(mc_bnb_evt_df.slc.vertex.columns)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import pandas as pd

# Standard definitions (assuming BINS_Z, BINS_Y, and sunset_cmap exist in your scope)
BINS_Z = np.linspace(0, 500, 100)
BINS_Y = np.linspace(-200, 200, 100)


def plot_split_tpc_2d(
    df: pd.DataFrame,
    pos_type: str = "vtx",
    weight_col: tuple = None,
    cmap: LinearSegmentedColormap = None,
    figsize: tuple = (15, 6),
):
    """Generates a 1x2 2D histogram of Y (vertical) vs Z (horizontal)

    split by X < 0 (TPC 0) and X >= 0 (TPC 1) using MultiIndex columns.
    """
    # Define tuple column mappings based on pos_type
    col_map = {
        "vtx": {
            "x": ("slc", "vertex", "x", "", "", ""),
            "y": ("slc", "vertex", "y", "", "", ""),
            "z": ("slc", "vertex", "z", "", "", ""),
        },
        # Add additional pos_types here if needed, e.g.:
        # "start": {
        #     "x": ("trk", "start", "x", "", "", ""),
        #     "y": ("trk", "start", "y", "", "", ""),
        #     "z": ("trk", "start", "z", "", "", ""),
        # },
    }

    if pos_type not in col_map:
        raise ValueError(
            f"Invalid pos_type '{pos_type}'. Supported types: {list(col_map.keys())}"
        )

    cols = col_map[pos_type]

    # Safely extract MultiIndex column data
    try:
        x_raw = df[cols["x"]]
        y_raw = df[cols["y"]]
        z_raw = df[cols["z"]]
    except KeyError as e:
        raise KeyError(
            f"Could not find MultiIndex coordinate column for pos_type='{pos_type}'. Missing column: {e}"
        )

    # Filter out NaNs across spatial coordinates
    mask = x_raw.notna() & y_raw.notna() & z_raw.notna()
    plot_df = df[mask]

    x_vals = plot_df[cols["x"]].to_numpy(dtype=float)
    y_vals = plot_df[cols["y"]].to_numpy(dtype=float)
    z_vals = plot_df[cols["z"]].to_numpy(dtype=float)

    # Resolve Weights
    if weight_col and weight_col in plot_df.columns:
        weights = plot_df[weight_col].fillna(1.0).to_numpy(dtype=float)
    else:
        weights = np.ones_like(x_vals, dtype=float)

    # Keep strictly finite values
    finite_mask = (
        np.isfinite(x_vals)
        & np.isfinite(y_vals)
        & np.isfinite(z_vals)
        & np.isfinite(weights)
    )
    x_vals = x_vals[finite_mask]
    y_vals = y_vals[finite_mask]
    z_vals = z_vals[finite_mask]
    weights = weights[finite_mask]

    # Split into Left TPC (X < 0 cm) and Right TPC (X >= 0 cm)
    mask_neg_x = x_vals < 0
    mask_pos_x = x_vals >= 0

    # Calculate global max bin count across both TPCs for consistent scaling
    h_left, _, _ = np.histogram2d(
        z_vals[mask_neg_x],
        y_vals[mask_neg_x],
        bins=[BINS_Z, BINS_Y],
        weights=weights[mask_neg_x],
    )
    h_right, _, _ = np.histogram2d(
        z_vals[mask_pos_x],
        y_vals[mask_pos_x],
        bins=[BINS_Z, BINS_Y],
        weights=weights[mask_pos_x],
    )

    vmax = max(
        h_left.max() if h_left.size > 0 else 0,
        h_right.max() if h_right.size > 0 else 0,
    )
    vmax = vmax if vmax > 0 else 1.0

    # Setup 1x2 Subplots
    fig, (ax_left, ax_right) = plt.subplots(
        1, 2, figsize=figsize, sharey=True, gridspec_kw={"wspace": 0.08}
    )

    # Default fallback cmap if none provided
    if cmap is None:
        cmap = plt.cm.viridis

    # --- Left Plot: X < 0 cm ---
    im0 = ax_left.hist2d(
        z_vals[mask_neg_x],
        y_vals[mask_neg_x],
        bins=[BINS_Z, BINS_Y],
        weights=weights[mask_neg_x],
        cmap=cmap,
        vmin=0,
        vmax=vmax,
    )[3]

    ax_left.set_title(
        f"{pos_type.upper()}: $X < 0$ cm ({mask_neg_x.sum()} hits)",
        fontsize=14,
        pad=10,
    )
    ax_left.set_xlabel("Z [cm]", fontsize=14)
    ax_left.set_ylabel("Y [cm]", fontsize=14)
    ax_left.set_xlim(BINS_Z[0], BINS_Z[-1])
    ax_left.set_ylim(BINS_Y[0], BINS_Y[-1])
    ax_left.grid(alpha=0.3, linestyle="--")

    # --- Right Plot: X >= 0 cm ---
    im1 = ax_right.hist2d(
        z_vals[mask_pos_x],
        y_vals[mask_pos_x],
        bins=[BINS_Z, BINS_Y],
        weights=weights[mask_pos_x],
        cmap=cmap,
        vmin=0,
        vmax=vmax,
    )[3]

    ax_right.set_title(
        f"{pos_type.upper()}: $X \\geq 0$ cm ({mask_pos_x.sum()} hits)",
        fontsize=14,
        pad=10,
    )
    ax_right.set_xlabel("Z [cm]", fontsize=14)
    ax_right.set_xlim(BINS_Z[0], BINS_Z[-1])
    ax_right.grid(alpha=0.3, linestyle="--")

    # Shared Colorbar
    cbar = fig.colorbar(im1, ax=[ax_left, ax_right], pad=0.02)
    cbar.set_label(
        "Weighted Entries" if weight_col else "Entries", fontsize=12
    )

    return fig, (ax_left, ax_right)

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')

    
print("data_tot_pot: %.3e" %(data_tot_pot))
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_bnb_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_evt_df))

In [ ]:

mask = build_event_cumulative_masks(mc_bnb_evt_df, sideband = "")

In [ ]:
'''
test_df = mc_bnb_evt_df[mask["t0"]]
fig, axes = plot_split_tpc_2d(
    test_df, pos_type="vtx", weight_col=pot_weight_col, cmap=sunset_cmap
)

test_df = mc_bnb_evt_df[mask["FV"]]
fig, axes = plot_split_tpc_2d(
    test_df, pos_type="vtx", weight_col=pot_weight_col, cmap=sunset_cmap
)


test_df = mc_bnb_evt_df[mask["proton_BDT"]]
fig, axes = plot_split_tpc_2d(
    test_df, pos_type="vtx", weight_col=pot_weight_col, cmap=sunset_cmap
)

test_df = mc_bnb_evt_df[mask["containment"]]
fig, axes = plot_split_tpc_2d(
    test_df, pos_type="vtx", weight_col=pot_weight_col, cmap=sunset_cmap
)


test_df = mc_bnb_evt_df[mask["TPC_containment"]]
fig, axes = plot_split_tpc_2d(
    test_df, pos_type="vtx", weight_col=pot_weight_col, cmap=sunset_cmap
)


test_df = mc_bnb_evt_df[mask["michel"]]
fig, axes = plot_split_tpc_2d(
    test_df, pos_type="vtx", weight_col=pot_weight_col, cmap=sunset_cmap
)
'''

test_df = mc_bnb_evt_df[mask["extra_pion"]]
fig, axes = plot_split_tpc_2d(
    test_df, pos_type="vtx", weight_col=pot_weight_col, cmap=sunset_cmap
)

final_mask = mask["extra_pion"] & (mc_bnb_evt_df > -10000)  #& (mc_bnb_evt_df.slc.measure_var.reco_p_mu > 0.1) #& (mc_bnb_evt_df.slc.measure_var.TLE_p_pi > 0.13) & (mc_bnb_evt_df.slc.measure_var.TLE_p_pi < 2)  
test_df = mc_bnb_evt_df[final_mask]
fig, axes = plot_split_tpc_2d(
    test_df, pos_type="vtx", weight_col=pot_weight_col, cmap=sunset_cmap
)     


In [ ]:
print(mc_bnb_evt_df[(mc_bnb_evt_df.slc.vertex.x > 0) & (mc_bnb_evt_df.slc.vertex.z > 250) & (mc_bnb_evt_df.slc.vertex.y > 100)].slc.measure_var.reco_p_mu)

